# Nairobi Flood Surrogate: Attention U-Net

The U-Net learns to **emulate a physics-based 2D flood simulator** (local inertial shallow water equations + SCS Curve Number runoff, `scripts/08_simulate_flood_scenarios.py`) so that a flood map for any storm can be produced in about a second instead of minutes.

* **Inputs (10 channels):** 7 terrain / land-cover / building channels (TPI, slope, HAND, log flow accumulation, depression depth, curve number, building coverage fraction) + 3 storm channels (intensity, duration, SCS runoff depth).
* **Outputs (3 heads):** flood probability (depth > 0.15 m), water depth (m) and depth x velocity (DV, m²/s, used by the vehicle-stability criteria of ARR Project 10).
* **Validation design:** geographic block split (train / validation / test blocks never overlap) **and** two storms that are never used in training (unseen-storm test).

**Before running:** upload `nairobi_surrogate_dataset.npz` to `MyDrive/Flood_Project/`, then choose *Runtime → Change runtime type → T4 GPU*. If the session disconnects, simply re-run all cells: training resumes from the last checkpoint saved on Drive.

In [ ]:
# 1. Setup: paths (Colab + Drive, or local for smoke tests)
import os, sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    OUT_DIR = '/content/drive/MyDrive/Flood_Project'
    DATA_PATH = '/content/nairobi_surrogate_dataset.npz'
    if not os.path.exists(DATA_PATH):
        shutil.copy(os.path.join(OUT_DIR, 'nairobi_surrogate_dataset.npz'), DATA_PATH)
else:
    DATA_PATH = os.environ.get('SURROGATE_DATA', 'scripts/data/surrogate/nairobi_surrogate_dataset.npz')
    OUT_DIR = os.environ.get('SURROGATE_OUT', 'scripts/models')

# SMOKE=1 runs a tiny version on CPU to check the code end-to-end
SMOKE = os.environ.get('SURROGATE_SMOKE') == '1'
os.makedirs(OUT_DIR, exist_ok=True)
print('Data:', DATA_PATH, '| Outputs:', OUT_DIR, '| Smoke test:', SMOKE)

In [ ]:
# 2. Imports, configuration and data loading
import json, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib
if not IN_COLAB:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'
print('Device:', device, torch.cuda.get_device_name(0) if USE_AMP else '')

CFG = dict(
    patch=128, batch_size=16, samples_per_epoch=3200, epochs=90, patience=25,
    lr=1e-3, weight_decay=1e-5, pos_weight=3.0, depth_loss_weight=1.0, dv_loss_weight=1.0, dv_scale=10.0,
    flood_depth_m=0.15, i_max=120.0, d_max=6.0, q_scale=100.0,
    infer_tile=512, infer_margin=32,
)
if SMOKE:
    CFG.update(samples_per_epoch=32, epochs=2, batch_size=4)

d = np.load(DATA_PATH)
static = torch.from_numpy(d['static'].astype(np.float32))   # (C, H, W) normalised
cn = d['cn'].astype(np.float32)                              # (H, W)
depth_np = d['depth']                                        # (S, H, W) float16, metres
dv_np = d['dv']                                              # (S, H, W) float16, max depth x velocity (m2/s)
storms = d['storms']                                         # (S, 2) intensity mm/hr, duration hr
split = d['split']                                           # 0 train, 1 val, 2 test, 255 border
holdout = d['holdout']
CHANNELS = [str(c) for c in d['channels']]
stats_mean, stats_std = d['stats_mean'], d['stats_std']
S, H, W = depth_np.shape
train_storms = [s for s in range(S) if not holdout[s]]
holdout_storms = [s for s in range(S) if holdout[s]]
print(f'Grid {H}x{W}, {S} storms ({len(holdout_storms)} held out), static channels: {CHANNELS}')

In [ ]:
# 3. Storm channels: intensity, duration and SCS-CN runoff depth (same formula as the simulator)
def scs_runoff_mm(p_mm, cn_map):
    s = 25400.0 / cn_map - 254.0
    ia = 0.2 * s
    return np.where(p_mm > ia, (p_mm - ia) ** 2 / (p_mm + 0.8 * s), 0.0).astype(np.float32)

def storm_channels(i_mmhr, d_hr):
    q = scs_runoff_mm(i_mmhr * d_hr, cn) / CFG['q_scale']
    return torch.stack([
        torch.full((H, W), i_mmhr / CFG['i_max']),
        torch.full((H, W), d_hr / CFG['d_max']),
        torch.from_numpy(q),
    ])

def build_input(i_mmhr, d_hr):
    # (C + 3, H, W) model input for the whole grid
    return torch.cat([static, storm_channels(i_mmhr, d_hr)], dim=0)

storm_inputs = [storm_channels(float(i), float(dh)).half() for i, dh in storms]
IN_CH = static.shape[0] + 3

In [ ]:
# 4. Training sampler: random 128x128 crops fully inside TRAIN blocks, biased towards wet areas
def valid_positions(code_, patch, stride):
    inside = (split == code_).astype(np.int32)
    ii = np.pad(inside.cumsum(0).cumsum(1), ((1, 0), (1, 0)))
    ys, xs = np.meshgrid(np.arange(0, H - patch + 1, stride), np.arange(0, W - patch + 1, stride), indexing='ij')
    ys, xs = ys.ravel(), xs.ravel()
    total = ii[ys + patch, xs + patch] - ii[ys, xs + patch] - ii[ys + patch, xs] + ii[ys, xs]
    keep = total == patch * patch
    return ys[keep], xs[keep]

P = CFG['patch']
pos_y, pos_x = valid_positions(0, P, 8)
print(f'{len(pos_y)} candidate training windows')

# Sampling weight per (storm, window) = 0.1 + fraction of flooded cells in the window
sample_w = {}
for s in train_storms:
    wet = (depth_np[s] > CFG['flood_depth_m']).astype(np.int32)
    ii = np.pad(wet.cumsum(0).cumsum(1), ((1, 0), (1, 0)))
    frac = (ii[pos_y + P, pos_x + P] - ii[pos_y, pos_x + P] - ii[pos_y + P, pos_x] + ii[pos_y, pos_x]) / (P * P)
    w = 0.1 + frac
    sample_w[s] = w / w.sum()

class TrainCrops(Dataset):
    def __len__(self):
        return CFG['samples_per_epoch']

    def __getitem__(self, _):
        s = random.choice(train_storms)
        k = np.random.choice(len(pos_y), p=sample_w[s])
        y0, x0 = pos_y[k], pos_x[k]
        x = torch.cat([static[:, y0:y0 + P, x0:x0 + P], storm_inputs[s][:, y0:y0 + P, x0:x0 + P].float()])
        # Target channels: 0 = max depth (m), 1 = max depth x velocity (m2/s)
        tgt = torch.from_numpy(np.stack([depth_np[s, y0:y0 + P, x0:x0 + P],
                                         dv_np[s, y0:y0 + P, x0:x0 + P]]).astype(np.float32))
        # Terrain physics is direction-independent and DV is a magnitude, so flips/rotations are valid
        if random.random() < 0.5:
            x, tgt = x.flip(-1), tgt.flip(-1)
        if random.random() < 0.5:
            x, tgt = x.flip(-2), tgt.flip(-2)
        k_rot = random.randint(0, 3)
        if k_rot:
            x, tgt = torch.rot90(x, k_rot, (1, 2)), torch.rot90(tgt, k_rot, (1, 2))
        return x.contiguous(), tgt.contiguous()

train_loader = DataLoader(TrainCrops(), batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=2 if IN_COLAB else 0, drop_last=True)

In [ ]:
# 5. Attention U-Net (same architecture as the original project, now terrain + building + storm inputs and 3 output heads)
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.net(x)

class AttentionBlock(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1), nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        return x * self.psi(self.relu(self.W_g(g) + self.W_x(x)))

class AttentionUNet(nn.Module):
    # Output channel 0 = flood logit, 1 = log(1 + depth in metres), 2 = log(1 + dv_scale * DV in m2/s)
    def __init__(self, in_channels=9, out_channels=2):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, 32); self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(32, 64);          self.pool2 = nn.MaxPool2d(2)
        self.enc3 = DoubleConv(64, 128);         self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(128, 256)
        self.dropout = nn.Dropout2d(p=0.3)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.att3 = AttentionBlock(128, 128, 64);  self.dec3 = DoubleConv(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.att2 = AttentionBlock(64, 64, 32);    self.dec2 = DoubleConv(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.att1 = AttentionBlock(32, 32, 16);    self.dec1 = DoubleConv(64, 32)
        self.final_conv = nn.Conv2d(32, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b = self.dropout(self.bottleneck(self.pool3(e3)))
        d3 = self.up3(b);  d3 = self.dec3(torch.cat([d3, self.att3(d3, e3)], 1))
        d2 = self.up2(d3); d2 = self.dec2(torch.cat([d2, self.att2(d2, e2)], 1))
        d1 = self.up1(d2); d1 = self.dec1(torch.cat([d1, self.att1(d1, e1)], 1))
        return self.final_conv(d1)

class SurrogateLoss(nn.Module):
    # Flood head: 0.4 weighted BCE + 0.6 Dice. Depth and DV heads: Smooth L1 on log-scaled targets.
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([CFG['pos_weight']], device=device))

    def forward(self, out, tgt):
        dep, dv = tgt[:, :1], tgt[:, 1:2]
        flood = (dep > CFG['flood_depth_m']).float()
        logit = out[:, :1]
        prob = torch.sigmoid(logit)
        inter = (prob * flood).sum()
        dice = 1 - (2 * inter + 1) / (prob.sum() + flood.sum() + 1)
        cls = 0.4 * self.bce(logit, flood) + 0.6 * dice
        reg = F.smooth_l1_loss(out[:, 1:2], torch.log1p(dep), beta=0.1)
        dvl = F.smooth_l1_loss(out[:, 2:3], torch.log1p(CFG['dv_scale'] * dv), beta=0.1)
        loss = cls + CFG['depth_loss_weight'] * reg + CFG['dv_loss_weight'] * dvl
        return loss, cls.item(), reg.item(), dvl.item()

OUT_CH = 3
model = AttentionUNet(IN_CH, OUT_CH).to(device)
print(f'{sum(p.numel() for p in model.parameters()) / 1e6:.2f} M parameters')

In [ ]:
# 6. Whole-grid inference (tiles with overlapping margins, so there are no seams)
@torch.no_grad()
def predict_full(net, i_mmhr, d_hr, temperature=1.0):
    net.eval()
    x = build_input(float(i_mmhr), float(d_hr))
    T, M = CFG['infer_tile'], CFG['infer_margin']
    logit_map = np.zeros((H, W), np.float32)
    depth_map = np.zeros((H, W), np.float32)
    dv_map = np.zeros((H, W), np.float32)
    ys = list(range(0, H - T, T - 2 * M)) + [H - T]
    xs = list(range(0, W - T, T - 2 * M)) + [W - T]
    for y0 in ys:
        for x0 in xs:
            with torch.autocast(device.type, dtype=torch.float16, enabled=USE_AMP):
                out = net(x[None, :, y0:y0 + T, x0:x0 + T].to(device)).float()[0].cpu().numpy()
            ya, yb = y0 + (M if y0 > 0 else 0), y0 + T - (M if y0 + T < H else 0)
            xa, xb = x0 + (M if x0 > 0 else 0), x0 + T - (M if x0 + T < W else 0)
            logit_map[ya:yb, xa:xb] = out[0, ya - y0:yb - y0, xa - x0:xb - x0]
            depth_map[ya:yb, xa:xb] = out[1, ya - y0:yb - y0, xa - x0:xb - x0]
            dv_map[ya:yb, xa:xb] = out[2, ya - y0:yb - y0, xa - x0:xb - x0]
    return (logit_map / temperature, np.expm1(np.clip(depth_map, 0, None)),
            np.expm1(np.clip(dv_map, 0, None)) / CFG['dv_scale'])

def collect(net, code_, storm_idx):
    # Model outputs and simulator truth for every cell of one split, per storm
    m = split == code_
    rows = []
    for s in storm_idx:
        logit, dep, dv = predict_full(net, *storms[s])
        rows.append(dict(storm=s, logit=logit[m], dep_pred=dep[m], dep_true=depth_np[s][m].astype(np.float32),
                         dv_pred=dv[m], dv_true=dv_np[s][m].astype(np.float32)))
    return rows

def iou_at(rows, thr=0.5, temperature=1.0):
    tp = fp = fn = 0
    for r in rows:
        p = 1 / (1 + np.exp(-r['logit'] / temperature)) >= thr
        t = r['dep_true'] > CFG['flood_depth_m']
        tp += np.sum(p & t); fp += np.sum(p & ~t); fn += np.sum(~p & t)
    return tp / max(tp + fp + fn, 1)

In [ ]:
# 7. Training with checkpoint/resume on Drive, selection by validation IoU
# Named by input/output count so an older checkpoint with a different architecture is never resumed
CKPT_PATH = os.path.join(OUT_DIR, f'surrogate_checkpoint_{IN_CH}in_{OUT_CH}out.pth')
BEST_PATH = os.path.join(OUT_DIR, 'unet_surrogate.pth')

criterion = SurrogateLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

start_epoch, best_iou, bad_epochs, history = 1, -1.0, 0, []
if os.path.exists(CKPT_PATH) and not SMOKE:
    # Our own checkpoint (it holds optimizer state and numpy scalars), so full unpickling is safe here
    ck = torch.load(CKPT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer'])
    scheduler.load_state_dict(ck['scheduler']); scaler.load_state_dict(ck['scaler'])
    start_epoch, best_iou, bad_epochs, history = ck['epoch'] + 1, ck['best_iou'], ck['bad_epochs'], ck['history']
    print(f'Resumed from epoch {ck["epoch"]} (best val IoU {best_iou:.4f})')

val_storms = train_storms[:2] if SMOKE else train_storms

for epoch in range(start_epoch, CFG['epochs'] + 1):
    if bad_epochs >= CFG['patience']:
        print('Early stopping.'); break
    t0 = time.time()
    model.train()
    tot = cls_tot = reg_tot = dv_tot = 0.0
    for xb, db in train_loader:
        xb, db = xb.to(device, non_blocking=True), db.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device.type, dtype=torch.float16, enabled=USE_AMP):
            out = model(xb)
        loss, c, r, v = criterion(out.float(), db)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        tot += loss.item(); cls_tot += c; reg_tot += r; dv_tot += v
    nb = len(train_loader)

    val_rows = collect(model, 1, val_storms)
    val_iou = iou_at(val_rows)
    scheduler.step(val_iou)
    history.append(dict(epoch=epoch, loss=tot / nb, cls=cls_tot / nb, depth=reg_tot / nb, dv=dv_tot / nb, val_iou=float(val_iou),
                        lr=optimizer.param_groups[0]['lr']))
    improved = val_iou > best_iou
    if improved:
        best_iou, bad_epochs = float(val_iou), 0
        torch.save(model.state_dict(), BEST_PATH)
    else:
        bad_epochs += 1
    torch.save(dict(model=model.state_dict(), optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
                    scaler=scaler.state_dict(), epoch=epoch, best_iou=float(best_iou), bad_epochs=bad_epochs,
                    history=history), CKPT_PATH)
    print(f'Epoch {epoch:03d} | loss {tot / nb:.4f} (flood {cls_tot / nb:.4f}, depth {reg_tot / nb:.4f}, DV {dv_tot / nb:.4f}) | '
          f'val IoU {val_iou:.4f}{"  *best*" if improved else ""} | {time.time() - t0:.0f}s')

model.load_state_dict(torch.load(BEST_PATH, map_location=device))
print(f'Loaded best model (val IoU {best_iou:.4f})')

In [ ]:
# 8. Learning curves
h = history
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot([r['epoch'] for r in h], [r['cls'] for r in h], label='flood loss')
ax[0].plot([r['epoch'] for r in h], [r['depth'] for r in h], label='depth loss')
ax[0].plot([r['epoch'] for r in h], [r['dv'] for r in h], label='DV loss')
ax[0].set_xlabel('Epoch'); ax[0].set_title('Training loss'); ax[0].legend()
ax[1].plot([r['epoch'] for r in h], [r['val_iou'] for r in h], color='#16a34a')
ax[1].set_xlabel('Epoch'); ax[1].set_title('Validation IoU (unseen geographic blocks)')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'surrogate_learning_curves.png'), dpi=200); plt.show()

In [ ]:
# 9. Calibration (temperature scaling) and decision threshold, both fitted on VALIDATION only
val_rows = collect(model, 1, val_storms)
rng = np.random.default_rng(SEED)
v_logit = np.concatenate([r['logit'] for r in val_rows])
v_true = np.concatenate([r['dep_true'] for r in val_rows]) > CFG['flood_depth_m']
idx = rng.choice(len(v_logit), size=min(len(v_logit), 1_000_000), replace=False)
lt = torch.tensor(v_logit[idx]); yt = torch.tensor(v_true[idx], dtype=torch.float32)

log_t = torch.zeros(1, requires_grad=True)
opt_t = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
def _closure():
    opt_t.zero_grad()
    l = F.binary_cross_entropy_with_logits(lt / log_t.exp(), yt)
    l.backward()
    return l
opt_t.step(_closure)
TEMPERATURE = float(log_t.exp().item())

thresholds = np.round(np.arange(0.10, 0.91, 0.05), 2)
sweep = []
for t in thresholds:
    p = 1 / (1 + np.exp(-v_logit / TEMPERATURE)) >= t
    tp, fp, fn = np.sum(p & v_true), np.sum(p & ~v_true), np.sum(~p & v_true)
    sweep.append((t, 2 * tp / max(2 * tp + fp + fn, 1), tp / max(tp + fp + fn, 1)))
THRESHOLD = float(max(sweep, key=lambda r: r[1])[0])
print(f'Temperature: {TEMPERATURE:.3f} | best validation threshold: {THRESHOLD:.2f}')

plt.figure(figsize=(7, 4))
plt.plot(thresholds, [r[1] for r in sweep], label='F1'); plt.plot(thresholds, [r[2] for r in sweep], label='IoU')
plt.axvline(THRESHOLD, color='r', ls='--', label=f'chosen {THRESHOLD:.2f}')
plt.xlabel('Probability threshold'); plt.title('Threshold selection (validation blocks)'); plt.legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'surrogate_threshold_sweep.png'), dpi=200); plt.show()

In [ ]:
# 10. TEST evaluation: unseen geographic blocks, for training storms and for held-out storms
from sklearn.metrics import average_precision_score

def metrics(rows, thr=None, temp=None):
    thr = THRESHOLD if thr is None else thr
    temp = TEMPERATURE if temp is None else temp
    logit = np.concatenate([r['logit'] for r in rows])
    prob = 1 / (1 + np.exp(-logit / temp))
    dt = np.concatenate([r['dep_true'] for r in rows])
    dp = np.concatenate([r['dep_pred'] for r in rows])
    t = dt > CFG['flood_depth_m']
    p = prob >= thr
    tp, fp, fn, tn = np.sum(p & t), np.sum(p & ~t), np.sum(~p & t), np.sum(~p & ~t)
    sub = np.random.default_rng(SEED).choice(len(t), size=min(len(t), 2_000_000), replace=False)
    wet = t
    return dict(
        IoU=tp / max(tp + fp + fn, 1), F1=2 * tp / max(2 * tp + fp + fn, 1),
        Precision=tp / max(tp + fp, 1), Recall=tp / max(tp + fn, 1),
        PR_AUC=average_precision_score(t[sub], prob[sub]) if t[sub].any() else float('nan'),
        Depth_MAE_wet_m=float(np.mean(np.abs(dp[wet] - dt[wet]))) if wet.any() else float('nan'),
        Depth_RMSE_all_m=float(np.sqrt(np.mean((dp - dt) ** 2))),
        DV_MAE_wet_m2s=float(np.mean(np.abs(np.concatenate([r['dv_pred'] for r in rows])[wet] -
                                            np.concatenate([r['dv_true'] for r in rows])[wet]))) if wet.any() else float('nan'),
        flooded_share=t.mean(), TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn))

test_seen = collect(model, 2, train_storms[:2] if SMOKE else train_storms)
test_unseen = collect(model, 2, holdout_storms[:1] if SMOKE else holdout_storms) if holdout_storms else []

results = {'test_blocks_training_storms': metrics(test_seen)}
if test_unseen:
    results['test_blocks_heldout_storms'] = metrics(test_unseen)

for name, m in results.items():
    print(f'\n== {name} ==')
    for k, v in m.items():
        print(f'  {k:18s} {v:.4f}' if isinstance(v, float) else f'  {k:18s} {v}')

print('\nPer-storm test IoU:')
for r in test_seen + test_unseen:
    i, dh = storms[r['storm']]
    m = metrics([r])
    tag = ' (held-out storm)' if holdout[r['storm']] else ''
    print(f'  {int(i):3d} mm/hr x {int(dh)} h: IoU {m["IoU"]:.3f}  F1 {m["F1"]:.3f}  depth MAE {m["Depth_MAE_wet_m"]:.3f} m{tag}')

In [ ]:
# 11. Baselines: simple terrain thresholds, tuned per storm on VALIDATION, scored on TEST
def raw(name):
    k = CHANNELS.index(name)
    return static[k].numpy() * stats_std[k] + stats_mean[k]

hand_raw, sink_raw = raw('hand'), raw('sink_depth')
mv, mt = split == 1, split == 2

def threshold_baseline(feature, lower_is_wet, storm_idx):
    tp = fp = fn = 0
    cands = np.quantile(feature[mv], np.linspace(0.01, 0.99, 60))
    for s in storm_idx:
        tv = depth_np[s][mv] > CFG['flood_depth_m']
        tt = depth_np[s][mt] > CFG['flood_depth_m']
        best, best_c = -1, cands[0]
        for c in cands:
            p = feature[mv] <= c if lower_is_wet else feature[mv] >= c
            iou = np.sum(p & tv) / max(np.sum(p | tv), 1)
            if iou > best:
                best, best_c = iou, c
        p = feature[mt] <= best_c if lower_is_wet else feature[mt] >= best_c
        tp += np.sum(p & tt); fp += np.sum(p & ~tt); fn += np.sum(~p & tt)
    return dict(IoU=tp / max(tp + fp + fn, 1), F1=2 * tp / max(2 * tp + fp + fn, 1))

eval_storms = train_storms[:2] if SMOKE else train_storms
baselines = {
    'HAND threshold': threshold_baseline(hand_raw, True, eval_storms),
    'Depression-depth threshold': threshold_baseline(sink_raw, False, eval_storms),
    'Attention U-Net (this work)': {k: results['test_blocks_training_storms'][k] for k in ('IoU', 'F1')},
}
print(f'{"Method":30s} {"IoU":>7s} {"F1":>7s}   (test blocks, training storms)')
for k, v in baselines.items():
    print(f'{k:30s} {v["IoU"]:7.4f} {v["F1"]:7.4f}')
results['baselines'] = baselines

In [ ]:
# 12. Reliability diagram (are the probabilities trustworthy?) - test blocks
logit = np.concatenate([r['logit'] for r in test_seen])
t = np.concatenate([r['dep_true'] for r in test_seen]) > CFG['flood_depth_m']
bins = np.linspace(0, 1, 11)

def reliability(prob):
    centers, acc, ece = [], [], 0.0
    for a, b in zip(bins[:-1], bins[1:]):
        sel = (prob >= a) & (prob < b)
        if sel.sum() == 0:
            continue
        centers.append(prob[sel].mean()); acc.append(t[sel].mean())
        ece += sel.mean() * abs(prob[sel].mean() - t[sel].mean())
    return centers, acc, ece

c_raw, a_raw, ece_raw = reliability(1 / (1 + np.exp(-logit)))
c_cal, a_cal, ece_cal = reliability(1 / (1 + np.exp(-logit / TEMPERATURE)))
results['calibration'] = dict(temperature=TEMPERATURE, ECE_before=float(ece_raw), ECE_after=float(ece_cal))
print(f'Expected calibration error: before {ece_raw:.4f} -> after temperature scaling {ece_cal:.4f}')

plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='perfect')
plt.plot(c_raw, a_raw, 'o-', label=f'raw (ECE {ece_raw:.3f})')
plt.plot(c_cal, a_cal, 's-', label=f'calibrated (ECE {ece_cal:.3f})')
plt.xlabel('Predicted flood probability'); plt.ylabel('Observed flood frequency'); plt.legend()
plt.title('Reliability diagram (test blocks)')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'surrogate_reliability.png'), dpi=200); plt.show()

In [ ]:
# 12b. Vehicle safety: does the U-Net make the same safe/unsafe call as the simulator, per vehicle class?
# Limits: ARR Project 10 (Shand, Cox, Blacka & Smith, 2011, report P10/S2/020), Table 3 - draft interim
# criteria for stationary vehicles: limiting still-water depth and constant depth x velocity (DV).
# Motorcycles and heavy vehicles are NOT covered by that report: the values below are labelled assumptions.
# A cell is unsafe if max depth > depth limit OR max DV > DV limit (the two maxima may occur at different
# times, so this is conservative). The 3 m/s velocity limit never binds here: the simulator's Froude cap keeps
# v <= sqrt(g h), which only exceeds 3 m/s above 0.9 m depth - already beyond every depth limit.
VEHICLES = {
    'boda_boda':  dict(label='Boda boda (motorcycle)', depth_m=CFG['flood_depth_m'], dv_m2s=None,
                       basis='Assumption: no published criterion; avoids any predicted flooding'),
    'saloon':     dict(label='Saloon car', depth_m=0.3, dv_m2s=0.3,
                       basis='ARR Project 10 Table 3: small passenger vehicle'),
    'large_car':  dict(label='Large car / matatu', depth_m=0.4, dv_m2s=0.45,
                       basis='ARR Project 10 Table 3: large passenger vehicle'),
    'suv_4x4':    dict(label='SUV / 4x4', depth_m=0.5, dv_m2s=0.6,
                       basis='ARR Project 10 Table 3: large 4WD'),
    'lorry_bus':  dict(label='Lorry / bus', depth_m=0.5, dv_m2s=0.6,
                       basis='Assumption: no published criterion for heavy vehicles; capped at the large 4WD limit'),
}

def unsafe(depth, dv, flooded, v):
    if v['dv_m2s'] is None:
        return flooded
    return flooded & ((depth > v['depth_m']) | (dv > v['dv_m2s']))

rows_v = test_seen + test_unseen
vehicle_results = {}
print(f'{"Vehicle":24s} {"unsafe share":>12s} {"Precision":>10s} {"Recall":>8s} {"F1":>7s}   (test blocks, all test storms)')
for key, v in VEHICLES.items():
    tp = fp = fn = n_unsafe = n = 0
    for r in rows_v:
        dt, dvt = r['dep_true'], r['dv_true']
        t = unsafe(dt, dvt, dt > CFG['flood_depth_m'], v)
        p = unsafe(r['dep_pred'], r['dv_pred'], 1 / (1 + np.exp(-r['logit'] / TEMPERATURE)) >= THRESHOLD, v)
        tp += np.sum(p & t); fp += np.sum(p & ~t); fn += np.sum(~p & t)
        n_unsafe += t.sum(); n += t.size
    m = dict(unsafe_share=n_unsafe / max(n, 1), Precision=tp / max(tp + fp, 1), Recall=tp / max(tp + fn, 1),
             F1=2 * tp / max(2 * tp + fp + fn, 1))
    vehicle_results[key] = m
    print(f'{v["label"]:24s} {m["unsafe_share"]:12.3f} {m["Precision"]:10.3f} {m["Recall"]:8.3f} {m["F1"]:7.3f}')
results['vehicles'] = vehicle_results


In [ ]:
# 13. Speed and maps: simulator vs U-Net for a held-out storm
s_show = holdout_storms[-1] if holdout_storms else train_storms[-1]
i_show, d_show = storms[s_show]
t0 = time.time()
logit_show, dep_show, dv_show = predict_full(model, i_show, d_show, TEMPERATURE)
infer_s = time.time() - t0
results['inference_seconds_full_grid'] = infer_s
print(f'U-Net whole-Nairobi prediction: {infer_s:.2f} s on {device}')

truth = depth_np[s_show].astype(np.float32)
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
vmax = 1.0
for a, img, title in [(ax[0], truth, 'Simulator depth (m)'), (ax[1], dep_show, 'U-Net depth (m)')]:
    im = a.imshow(np.where(img > 0.05, img, np.nan), cmap='Blues', vmin=0, vmax=vmax)
    a.contour(split == 2, levels=[0.5], colors='red', linewidths=0.8)
    a.set_title(f'{title} - {int(i_show)} mm/hr x {int(d_show)} h'); a.axis('off')
    plt.colorbar(im, ax=a, fraction=0.03)
im = ax[2].imshow(dep_show - truth, cmap='RdBu_r', vmin=-0.5, vmax=0.5)
ax[2].set_title('U-Net minus simulator (m); red = test blocks'); ax[2].axis('off')
ax[2].contour(split == 2, levels=[0.5], colors='red', linewidths=0.8)
plt.colorbar(im, ax=ax[2], fraction=0.03)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'surrogate_map_comparison.png'), dpi=200); plt.show()

In [ ]:
# 14. Export model + config for the FastAPI server (download both into scripts/models/)
def _clean(o):
    if isinstance(o, dict):
        return {k: _clean(v) for k, v in o.items()}
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    return o

config = dict(
    architecture='AttentionUNet', in_channels=IN_CH, out_channels=OUT_CH,
    outputs=['flood_logit', 'log1p_depth_m', 'log1p(dv_scale*DV_m2s)'], dv_scale=CFG['dv_scale'],
    vehicles=VEHICLES,
    static_channels=CHANNELS, storm_channels=['intensity/i_max', 'duration/d_max', 'scs_runoff_mm/q_scale'],
    stats_mean=[float(v) for v in stats_mean], stats_std=[float(v) for v in stats_std],
    i_max=CFG['i_max'], d_max=CFG['d_max'], q_scale=CFG['q_scale'], flood_depth_m=CFG['flood_depth_m'],
    temperature=TEMPERATURE, threshold=THRESHOLD, infer_tile=CFG['infer_tile'], infer_margin=CFG['infer_margin'],
    trained_storms=[[float(a), float(b)] for a, b in storms[train_storms]],
    heldout_storms=[[float(a), float(b)] for a, b in storms[holdout_storms]],
    best_val_iou=float(best_iou), metrics=_clean(results), train_config=CFG,
)
with open(os.path.join(OUT_DIR, 'unet_surrogate_config.json'), 'w') as f:
    json.dump(config, f, indent=2)
print('Saved unet_surrogate.pth and unet_surrogate_config.json to', OUT_DIR)

if IN_COLAB:
    from google.colab import files
    for fn in ['unet_surrogate.pth', 'unet_surrogate_config.json', 'surrogate_learning_curves.png',
               'surrogate_threshold_sweep.png', 'surrogate_reliability.png', 'surrogate_map_comparison.png']:
        files.download(os.path.join(OUT_DIR, fn))